In [0]:
# Let us get the count of orders from bronze layer dataset 
(
    spark.read.table("dev.multiplex_bronze.kafka_bronze")
            .filter("topic='orders'")
            .count()
)

In [0]:
# Drop duplicated records: from BatchRead 
from pyspark.sql import functions as F
json_schema = f"""
    order_id STRING,
    order_timestamp TIMESTAMP,
    customer_id STRING,
    quantity BIGINT,
    total BIGINT,
    books ARRAY<STRUCT<
        book_id: STRING,
        quantity: LONG,
        subtotal: DOUBLE>>
"""

dedup_batch = (
    spark.read.table("dev.multiplex_bronze.kafka_bronze")
            .filter(F.col("topic")=='orders')
            .select( 
                F.from_json(
                    F.col("value").cast("string"), 
                    json_schema
                ).alias("v")
            )
            .select("v.*")
            .dropDuplicates(["order_id", 'order_timestamp'])
            .count()
)

print(dedup_batch)

In [0]:
# Drop duplicated records: from StreamRead 
from pyspark.sql import functions as F
json_schema = f"""
    order_id STRING,
    order_timestamp TIMESTAMP,
    customer_id STRING,
    quantity BIGINT,
    total BIGINT,
    books ARRAY<STRUCT<
        book_id: STRING,
        quantity: BIGINT,
        subtotal: DOUBLE>>
"""

dedup_stream = (
    spark.readStream.table('dev.multiplex_bronze.kafka_bronze')
        .filter(F.col("topic") == 'orders')
        .select(
            F.from_json(
                F.col('value').cast('string'),
                schema= json_schema
            ).alias('v')        
        ).select('v.*')
        .withWatermark('order_timestamp', "30 seconds")
        .dropDuplicates(['order_id', 'order_timestamp'])
)

In [0]:
def upsert_data(microBatchDF, batch):
    microBatchDF.createOrReplaceTempView('orders_microbatch')

    sql_query = """
        MERGE INTO dev.silver.orders_silver o
        USING orders_microbatch b
        ON o.order_id = b.order_id
        WHEN NOT MATCHED THEN INSERT *
    """

    microBatchDF.sparkSession.sql(sql_query) # For clusters with runtime 11.3 or above 
    #microBatchDF._jdf.sparkSession().sql(sql_query) # For clusters with runtime 10.5 or below

In [0]:
%sql
-- CREATE a orders table for place holder
CREATE TABLE IF NOT EXISTS dev.silver.orders_table(
  order_id STRING,
  order_timestamp TIMESTAMP,
  customer_id STRING,
  quantity BIGINT,
  total BIGINT,
  books ARRAY<STRUCT<
      book_id STRING,
      quantity STRING,
      subtotal BIGINT>>
)

In [0]:
query_ = (
    dedup_stream.writeStream
        .foreachBatch(upsert_data)
        .option("checkpointLocation", "/Volumes/dev/pro_landing_zone/checkpoints/orders_table/")
        .trigger(availableNow=True)
        .table("dev.silver.orders_table")
)
query_.awaitTermination()

In [0]:
%sql
select * from dev.silver.orders_table